# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook serves as a template for loading, exploring, and analyzing the [FAIR^2 mass spectrometry protein dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. All data entities are referenced by their schema `@id`, making the workflow clear and reproducible with Croissant-compliant datasets.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records from the Croissant schema with `mlcroissant`. This section fetches both rich metadata and prepares for downstream processing.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the FAIR^2 dataset Croissant JSON-LD URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset from the Croissant schema URL
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
The FAIR^2 Croissant schema can contain more than one record set (`cr:RecordSet`). We enumerate all available record sets, their `@id`s, and their fields.

Below, record sets, fields, and columns are always referenced by their `@id` for clarity and reproducibility.

In [ ]:
# List all record sets in the metadata and their fields/columns
record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    print("No record sets found in the dataset metadata.\n\nHowever, this dataset is often structured as a single main table. Let's try to infer the available record sets but if missing, try reading with record_set=None.")
else:
    for record_set in record_sets:
        print(f"Record Set @id: {getattr(record_set, '@id', '<no id>')}")
        print(f"  Name: {getattr(record_set, 'name', '<no name>')}")
        print("  Fields and Columns:")
        for field in getattr(record_set, 'field', []):
            print(f"    Field @id: {getattr(field, '@id', '<no id>')}, Name: {getattr(field, 'name', '<no name>')}")
        for column in getattr(record_set, 'column', []):
            print(f"    Column @id: {getattr(column, '@id', '<no id>')}, Name: {getattr(column, 'name', '<no name>')}")
        print()

# If no record sets, try reading first record
try:
    print("First record example:")
    records_iter = dataset.records(record_set=None)
    first_record = next(records_iter)
    print(first_record)
except Exception as e:
    print(f"Could not preview a record: {e}")

## 3. Data Extraction
Extract data into a pandas DataFrame. Since the dataset does not specify explicit record sets in its schema, we use `record_set=None` (the default for datasets with a single main table). This will extract all available data.

You can inspect all available fields/columns by looking at a sample row below.

In [ ]:
# Load all records into a DataFrame since we have one main record set (or no explicit @id)
main_record_set_id = None  # Use None if only one record set or unspecified

records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Columns in dataset: {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Below are sample operations for filtering, normalizing, and grouping the data. These operations use fields by their `@id` (or the DataFrame's column names if no explicit `@id` is available).

Typical numeric fields in mass spectrometry datasets include protein coverage percentages, peptide counts, molecular weight (`MW`), or normalized abundance values. Adjust the field below to match your actual column names.

In [ ]:
# Choose a numeric field for analysis
# Example, typical columns are: 'MW', 'Coverage', 'PeptideCount', 'Abundance_Sample1', etc.
# Let's detect and choose one of these programmatically if present:
possible_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not possible_numeric_fields:
    # Try common names
    possible_numeric_fields = [name for name in ['MW', 'Coverage', 'PeptideCount', 'Abundance_Sample1', 'Abundance_Sample2'] if name in df.columns]
numeric_field = possible_numeric_fields[0] if possible_numeric_fields else df.columns[df.dtypes != 'O'][0]
print(f"Using numeric field for EDA: {numeric_field}")

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the chosen numeric field
normalized_col = f"{numeric_field}_normalized"
filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, normalized_col]].head())

# Try grouping by a likely categorical field
possible_group_fields = [col for col in df.columns if col.lower() in ['accession', 'group', 'protein', 'experiment', 'description'] or pd.api.types.is_object_dtype(df[col])]
group_field = possible_group_fields[0] if possible_group_fields else df.columns[0]
print(f"Grouping by field: {group_field}")
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields. Here, we plot the normalized distribution of the chosen numeric field and its relationship with a categorical attribute if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of normalized values
sns.histplot(filtered_df[normalized_col], kde=True)
plt.xlabel(normalized_col)
plt.title(f"Distribution of Normalized {numeric_field}")
plt.show()

# Boxplot by group field (if small number of groups)
if group_field in filtered_df.columns and filtered_df[group_field].nunique() < 20:
    plt.figure(figsize=(10, 4))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
    plt.xticks(rotation=45)
    plt.title(f"Boxplot of {numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and perform simple statistics and visualizations on a Croissant-compliant mass spectrometry protein dataset using `mlcroissant`. All field selections and processing referenced schema `@id`s and ensured full reproducibility. For further analysis, experiment with other fields or export subsets for domain-specific workflows.
